# MSC Figure 6: Middleton localization-confidence reproducibility bundle

Upload the validated Middleton localization-confidence source script, the MSC model, and its YAML configuration. This wrapper removes only Google Drive dependencies and writes an auditable local bundle plus ZIP.

The analysis retains the locked design: GSE115480 paired soma-dendrite data, localization-confidence selection, 250 dendritic plus 250 matched soma genes, Ensembl 116 / GRCm39, sequence scoring, PSD mapping, and MSC simulations.


## 1. Upload files and prepare the environment


In [ ]:
import subprocess
import sys
import hashlib
import json
import shutil
import zipfile
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'statsmodels', 'scipy', 'pandas', 'numpy', 'requests'], check=False)

from google.colab import files

WORK = Path('/content/msc_figure6_middleton')
WORK.mkdir(exist_ok=True)

print('Upload three files: localization-confidence .py source, MSC model .py, YAML config.')
uploaded = files.upload()

for filename in uploaded:
    shutil.move('/content/' + filename, WORK / filename)

source_files = list(WORK.glob('*localization_confidence*100seed.py'))
model_files = list(WORK.glob('msc_model_feedback*.py'))
config_files = list(WORK.glob('*.yaml')) + list(WORK.glob('*.yml'))

assert len(source_files) == 1, 'Upload msc_figure6_middleton_localization_confidence_100seed.py'
assert len(model_files) >= 1, 'Upload the MSC model .py file'
assert len(config_files) >= 1, 'Upload the YAML configuration file'

SOURCE = source_files[0]
MODEL = WORK / 'msc_model_feedback.py'
CONFIG = WORK / 'config_msc_feedback.yaml'

if model_files[0] != MODEL:
    shutil.copy2(model_files[0], MODEL)
if config_files[0] != CONFIG:
    shutil.copy2(config_files[0], CONFIG)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

print('Source hash:', sha256_file(SOURCE)[:16])
print('Model hash: ', sha256_file(MODEL)[:16])
print('Config hash:', sha256_file(CONFIG)[:16])


## 2. Create an explicit no-Drive copy of the validated source

The original source is retained unchanged. The copied script is redirected from Google Drive to the local Colab workspace. Public data and annotation downloads remain unchanged.


In [ ]:
source_text = SOURCE.read_text(encoding='utf-8')

# Colab-exported Python source may contain shell-magics beginning with !.
# Dependencies were installed in Cell 1, so these lines can be removed safely.
source_text = '\n'.join(line for line in source_text.splitlines() if not line.lstrip().startswith('!'))

source_text = source_text.replace('from google.colab import drive, files', 'from google.colab import files')
source_text = source_text.replace("drive.mount('/content/drive')", '')

old_root = "ROOT=Path('/content/drive/MyDrive/MSC_Results_0726/figure6_middleton_localization_confidence_v2')"
new_root = "ROOT=Path('/content/msc_figure6_middleton/output')"
assert old_root in source_text, 'Expected Middleton Drive output path was not found.'
source_text = source_text.replace(old_root, new_root)

old_model_line = "DRIVE_MODEL=Path('/content/drive/MyDrive/MSC_Results_0726/msc_model_feedback.py'); DRIVE_CFG=Path('/content/drive/MyDrive/MSC_Results_0726/config_msc_feedback.yaml')"
new_model_line = "DRIVE_MODEL=Path('/content/msc_figure6_middleton/msc_model_feedback.py'); DRIVE_CFG=Path('/content/msc_figure6_middleton/config_msc_feedback.yaml')"
assert old_model_line in source_text, 'Expected Middleton model/config Drive path was not found.'
source_text = source_text.replace(old_model_line, new_model_line)

PATCHED = WORK / 'msc_figure6_middleton_localization_confidence_100seed_nodrive.py'
PATCHED.write_text(source_text, encoding='utf-8')

assert '/content/drive/' not in source_text
assert 'MIN_DETECTION_CELLS=5' in source_text
assert 'MIN_DIRECTION_PAIRS=10' in source_text
assert 'N_MATCHED_PER_GROUP=250' in source_text
assert "ENSEMBL_RELEASE=116" in source_text
assert "ASSEMBLY='GRCm39'" in source_text

print('No-Drive source created and validated:', PATCHED.name)
print('Patched source hash:', sha256_file(PATCHED)[:16])


## 3. Run the complete Middleton workflow

This is the long cell. The workflow downloads public data and Ensembl annotation, builds the frozen panel before calculating sequence features, runs the PSC mappings, and writes checkpointed simulation outputs.


In [ ]:
print('Running validated Middleton localization-confidence workflow...')
workflow_namespace = {'__name__': '__main__', '__file__': str(PATCHED)}
exec(compile(source_text, str(PATCHED), 'exec'), workflow_namespace)

ROOT = Path('/content/msc_figure6_middleton/output')
assert ROOT.exists(), 'Expected output root was not created.'
print('Workflow completed:', ROOT)


## 4. Validate final tables, write manifest, and download ZIP


In [ ]:
import pandas as pd

canonical_names = [
    'panel_B_middleton_sequence_effects.csv',
    'panel_C_middleton_state_preferences.csv',
    'panel_D_middleton_wt_me_shifts.csv',
    'panel_D_middleton_mapping_contrasts.csv'
]

found = {}
for name in canonical_names:
    matches = list(ROOT.rglob(name))
    assert len(matches) == 1, f'Expected exactly one {name}; found {len(matches)}'
    found[name] = matches[0]

sequence_effects = pd.read_csv(found['panel_B_middleton_sequence_effects.csv'])
state_preferences = pd.read_csv(found['panel_C_middleton_state_preferences.csv'])
wt_me_shifts = pd.read_csv(found['panel_D_middleton_wt_me_shifts.csv'])
mapping_contrasts = pd.read_csv(found['panel_D_middleton_mapping_contrasts.csv'])

assert set(sequence_effects['dataset']) == {'Middleton_2019_GSE115480'}
assert len(sequence_effects) == 9
assert set(sequence_effects['n']) == {250}
assert set(state_preferences['n_seeds']) == {30, 100}
assert set(wt_me_shifts['n']) == {30, 100}
assert set(mapping_contrasts['n']) == {30, 100}
assert {'primary_g_rg4', 'g_only', 'primary_mapping_shuffled'}.issubset(set(wt_me_shifts['analysis_name']))

R_INPUT = ROOT / '10_r_input'
R_INPUT.mkdir(exist_ok=True)
for name, path in found.items():
    shutil.copy2(path, R_INPUT / name)

for extra in ['middleton_frozen_matched_panel.csv', 'middleton_matching_balance.csv', 'middleton_pair_diagnostics.csv']:
    matches = list(ROOT.rglob(extra))
    if len(matches) == 1:
        shutil.copy2(matches[0], R_INPUT / extra)

manifest = {
    'dataset': 'Middleton_2019_GSE115480',
    'pipeline': 'localization-confidence',
    'source_sha256': sha256_file(SOURCE),
    'patched_source_sha256': sha256_file(PATCHED),
    'model_sha256': sha256_file(MODEL),
    'config_sha256': sha256_file(CONFIG),
    'validation': {
        'sequence_effect_rows': int(len(sequence_effects)),
        'matched_pairs': 250,
        'state_seed_counts': sorted(state_preferences['n_seeds'].unique().tolist()),
        'shift_seed_counts': sorted(wt_me_shifts['n'].unique().tolist())
    },
    'outputs': {str(path.relative_to(ROOT)): sha256_file(path) for path in ROOT.rglob('*') if path.is_file()}
}

(ROOT / 'MANIFEST_middleton.json').write_text(json.dumps(manifest, indent=2))
(ROOT / 'README.md').write_text('# MSC Figure 6 Middleton bundle\n\nLocalization-confidence workflow using GSE115480 paired soma-dendrite counts. The frozen panel contains 250 dendritic and 250 matched soma genes. The primary tau2 PSD mapping uses 100 seeds; elastic modulus and mobile-fraction mappings use 30 seeds each.\n')

zip_path = Path('/content/MSC_Figure6_Middleton_reproducible_bundle.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in ROOT.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(ROOT.parent))

print('Validation passed.')
print('ZIP:', zip_path.name, f'{zip_path.stat().st_size / 1e6:.1f} MB')
display(sequence_effects)
files.download(str(zip_path))
